# SP广告数据分析工具 - 快速验证

这个Notebook用于快速验证数据提取和分析效果。

## 使用步骤：
1. 修改下面的 `excel_path` 为你的数据文件路径
2. 按 Shift+Enter 逐个运行每个单元格
3. 查看结果

In [ ]:
# 导入必要的库
import pandas as pd
import numpy as np
from pathlib import Path
from data_processor import (
    extract_all_dimensions,
    validate_excel,
    aggregate_single,
    aggregate_cross_multi_metrics,
    aggregate_cross,
    get_dimension_summary
)

print("✅ 库导入成功")

In [ ]:
# 修改这里为你的Excel文件路径
excel_path = '/Users/linshaoyong/Desktop/SP 数据分析/Acpatur-US-US等1个店铺-全部-1010-1108-汇总-846681677169868800.xlsx'

print(f"📁 数据文件：{excel_path}")

# 读取Excel
df = pd.read_excel(excel_path)
print(f"\n✅ 读取成功")
print(f"   - 行数：{len(df)}")
print(f"   - 列数：{len(df.columns)}")

In [ ]:
# 数据验证
validation = validate_excel(df)
print("✅ 数据验证结果：")
print(f"   状态：{'有效 ✓' if validation['valid'] else '无效 ✗'}")

if validation['errors']:
    print("\n❌ 错误：")
    for error in validation['errors']:
        print(f"   - {error}")

if validation['warnings']:
    print("\n⚠️ 警告：")
    for warning in validation['warnings']:
        print(f"   - {warning}")

In [ ]:
# 提取维度
print("⏳ 正在提取维度...")
df_extracted = extract_all_dimensions(df)
print("✅ 维度提取完成")

# 显示前10行
print("\n📋 前10行数据：")
display_cols = ['广告活动', 'parent_code', 'pattern', 'attribute', 'is_valid']
df_extracted[display_cols].head(10)

In [ ]:
# 维度统计
summary = get_dimension_summary(df_extracted)

print("📈 维度统计摘要")
print("=" * 60)
print(f"\n总行数：{summary['total_rows']}")
print(f"有效数据：{summary['total_rows'] - summary['invalid_count']} 行 ({100*(summary['total_rows']-summary['invalid_count'])/summary['total_rows']:.1f}%)")
print(f"异常数据：{summary['invalid_count']} 行 ({100*summary['invalid_count']/summary['total_rows']:.1f}%)")

print(f"\n📍 Parent Code：{summary['parent_codes']['count']} 种")
print(f"   {', '.join(summary['parent_codes']['values'][:10])}")
if len(summary['parent_codes']['values']) > 10:
    print(f"   ... 等共{summary['parent_codes']['count']}种")

print(f"\n🎨 图案：{summary['patterns']['count']} 种")
print(f"   {', '.join(summary['patterns']['values'][:10])}")
if len(summary['patterns']['values']) > 10:
    print(f"   ... 等共{summary['patterns']['count']}种")

print(f"\n🏷️ 属性：{summary['attributes']['count']} 种")
print(f"   {', '.join(summary['attributes']['values'][:10])}")
if len(summary['attributes']['values']) > 10:
    print(f"   ... 等共{summary['attributes']['count']}种")

In [ ]:
# 异常数据检查
invalid_data = df_extracted[~df_extracted['is_valid']]

if len(invalid_data) > 0:
    print(f"⚠️ 异常数据（共{len(invalid_data)}行）")
    print("=" * 60)
    display_cols = ['广告活动', 'parent_code', 'pattern', 'attribute']
    display(invalid_data[display_cols])
else:
    print("✅ 所有数据都有效")

In [ ]:
# 【示例1】单维度聚合 - 按 Parent Code 聚合
print("📊 示例1：按 Parent Code 聚合")
print("=" * 60)

result = aggregate_single(df_extracted, 'parent_code')

# 显示前15行
result_sorted = result.sort_values('花费', ascending=False)
print(f"\n共{len(result)}个Parent Code\n")
display(result_sorted.head(15))

In [ ]:
# 【示例2】单维度聚合 - 按 图案 聚合
print("📊 示例2：按 图案 聚合")
print("=" * 60)

result = aggregate_single(df_extracted, 'pattern')

# 显示前15行
result_sorted = result.sort_values('花费', ascending=False)
print(f"\n共{len(result)}个图案\n")
display(result_sorted.head(15))

In [ ]:
# 【示例3】交叉分析 - Parent Code × 属性（多指标）
print("📊 示例3：交叉分析 - Parent Code × 属性")
print("=" * 60)
print("\n显示多个指标（花费和销售额）\n")

result = aggregate_cross_multi_metrics(
    df_extracted,
    'parent_code',
    'attribute',
    ['花费', '销售额', 'ROAS']
)

display(result.head(10))

In [ ]:
# 【示例4】带筛选的交叉分析
print("📊 示例4：筛选Parent Code=City，交叉分析 图案 × 属性")
print("=" * 60)

filters = {'parent_code': ['City']}
result = aggregate_cross_multi_metrics(
    df_extracted,
    'pattern',
    'attribute',
    ['花费', '销售额'],
    filters
)

print(f"\n结果行数：{len(result)}\n")
display(result.head(20))

In [ ]:
# 【可视化】Parent Code 的花费对比
import matplotlib.pyplot as plt

result = aggregate_single(df_extracted, 'parent_code')
result_sorted = result.sort_values('花费', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(result_sorted['parent_code'], result_sorted['花费'], color='steelblue')
ax.set_xlabel('花费 ($)', fontsize=12)
ax.set_title('Top 15 Parent Code 花费对比', fontsize=14, fontweight='bold')
ax.invert_yaxis()

# 添加数值标签
for i, v in enumerate(result_sorted['花费']):
    ax.text(v, i, f' ${v:.0f}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

print("✅ 图表展示完成")

In [ ]:
# 导出结果到Excel
import datetime

output_path = Path(excel_path).parent / f"分析结果_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.xlsx"

print(f"💾 导出结果到 Excel...")
print(f"\n目标文件：{output_path}")

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    # 工作表1：提取的维度
    df_extracted[['广告活动', 'parent_code', 'pattern', 'attribute', 'is_valid']].to_excel(
        writer, sheet_name='提取的维度', index=False
    )

    # 工作表2：按Parent Code聚合
    agg_pc = aggregate_single(df_extracted, 'parent_code').sort_values('花费', ascending=False)
    agg_pc.to_excel(writer, sheet_name='Parent Code聚合', index=False)

    # 工作表3：按图案聚合
    agg_pattern = aggregate_single(df_extracted, 'pattern').sort_values('花费', ascending=False)
    agg_pattern.to_excel(writer, sheet_name='图案聚合', index=False)

    # 工作表4：交叉分析示例
    cross = aggregate_cross_multi_metrics(
        df_extracted, 'parent_code', 'pattern',
        ['花费', '销售额']
    )
    cross.to_excel(writer, sheet_name='交叉分析示例')

print(f"✅ 导出完成！")
print(f"\n包含 4 个工作表：")
print(f"  1. 提取的维度 - 原始数据+提取结果")
print(f"  2. Parent Code聚合 - 按Parent Code统计")
print(f"  3. 图案聚合 - 按图案统计")
print(f"  4. 交叉分析示例 - Parent Code × 图案")

## 总结

✅ 数据提取模块已完成验证，可以：
- ✅ 准确提取 3 个维度
- ✅ 处理各种异常情况
- ✅ 进行单维度和交叉分析
- ✅ 支持多指标同时显示
- ✅ 支持数据筛选和导出

下一步：进入 **第2阶段 - Streamlit Web界面开发** 🚀